### Import Libraries

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, max_error
from scipy.stats import median_abs_deviation
from tabulate import tabulate
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import VarianceThreshold

### Load the Datasets

In [2]:
#combined_df = pd.read_csv('combined_Octoparse_categorized.csv')
combined_df = pd.read_csv('../../Cleaned Data/Cleaned_Data_LastVersion.csv')
target_variable = 'Number of Reviewers'
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1854 entries, 0 to 1853
Data columns (total 48 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Business ID                               1854 non-null   object 
 1   Name                                      1854 non-null   object 
 2   Latitude                                  1854 non-null   float64
 3   Longitude                                 1854 non-null   float64
 4   Category                                  1854 non-null   object 
 5   Rating                                    1854 non-null   object 
 6   Popularity                                1854 non-null   float64
 7   Google Place ID                           1854 non-null   object 
 8   Business Status                           1854 non-null   object 
 9   Distance (m)                              1854 non-null   float64
 10  Cluster                             

## Data Pre-processing

### Drop Irrelevent columns

In [3]:
# Drop irrelevant columns
drop_columns = ['Business ID', 'Latitude'	, 'Longitude', 'Name', 'Category', 'Rating', 'Google Place ID', 'Business Status',
                'Cluster', 'Distance (m)', 'Avg Rating - Business Type', 'Competition - Business Type/Area', 'Competition - Food & Dining/Area',
                'Competition - Business Type/POI Density', 'Competition - Food & Dining/related POIs', 'Competition - Business Type/related POIs'
               , 'Google Rating', 'Business Name']

combined_df.drop(columns=drop_columns, inplace=True, errors='ignore')


### Drop Rows with missing #of Reviewrs

In [4]:
# Ensure that the 'Number of Reviewers' column is numeric and handle errors as NaN
combined_df[target_variable] = pd.to_numeric(combined_df[target_variable], errors='coerce')

# Drop rows where 'Number of Reviewers' is NaN (if you want to ignore them in analysis)
combined_df = combined_df.dropna(subset=[target_variable])

### Fill Missing Values with median

In [5]:
print(combined_df.isnull().sum())

Popularity                                  0
generalCategory                             0
Religious Institutions                      0
Coffee Shops                                0
Food & Dining                               0
Restaurants                                 0
Home & Construction Services                0
Entertainment & Recreation                  0
Retail & Shopping                           0
Finance & Services                          0
Education                                   0
Health                                      0
Public & Government Services                0
Hotels & Hospitality                        0
Transportation & Travel                     0
Beauty & Wellness                           0
POI Density                                 0
Avg Rating - Food & Dining                 15
Competition - Food & Dining/POI Density     0
Population Within 1km                       0
Number of Reviewers                         0
Avg_NumOfReviewers - Food & Dining

In [5]:
#Fill missing values using median of the specfic food-chain

combined_df["Avg Rating - Food & Dining"] = combined_df["Avg Rating - Food & Dining"].fillna(combined_df["Avg Rating - Food & Dining"].median())
combined_df["Avg_NumOfReviewers - Food & Dining"] = combined_df["Avg_NumOfReviewers - Food & Dining"].fillna(combined_df["Avg_NumOfReviewers - Food & Dining"].median())
combined_df["Avg_NumOfReviewers_SameCategory"] = combined_df["Avg_NumOfReviewers_SameCategory"].fillna(combined_df["Avg_NumOfReviewers_SameCategory"].median())
combined_df["Total Num of Reviewers - Food & Dining"] = combined_df["Total Num of Reviewers - Food & Dining"].fillna(combined_df["Total Num of Reviewers - Food & Dining"].median())

print(combined_df.isnull().sum())

Popularity                                 0
generalCategory                            0
Religious Institutions                     0
Coffee Shops                               0
Food & Dining                              0
Restaurants                                0
Home & Construction Services               0
Entertainment & Recreation                 0
Retail & Shopping                          0
Finance & Services                         0
Education                                  0
Health                                     0
Public & Government Services               0
Hotels & Hospitality                       0
Transportation & Travel                    0
Beauty & Wellness                          0
POI Density                                0
Avg Rating - Food & Dining                 0
Competition - Food & Dining/POI Density    0
Population Within 1km                      0
Number of Reviewers                        0
Avg_NumOfReviewers - Food & Dining         0
Avg_NumOfR

## Dataset Splitting

In [6]:
# --- Training and Testing ----

# Split combined_df into 80% train and 20% test
districts = combined_df['District'].unique()

train_districts, test_districts  = train_test_split(
    #combined_df,
    districts,
    test_size=0.2,
    random_state=42,  # for reproducibility
    shuffle=True      # ensure data is shuffled before splitting
)

combined_df_train = combined_df[combined_df['District'].isin(train_districts)]
combined_df_test = combined_df[combined_df['District'].isin(test_districts)]

# Optional: Print shape summary
print(f"Train set size: {combined_df_train.shape[0]}")
print(f"Test set size: {combined_df_test.shape[0]}")


Train set size: 1493
Test set size: 359


### X and y Variables

In [7]:
# -----------------------------
# Step 1: Define features and target
# -----------------------------

# Columns to drop

columns_to_drop = [target_variable, 'Total Num of Reviewers - Food & Dining', 'Total Num of Reviewers - All POIs', 
                   'Avg_NumOfReviewers - Food & Dining','Avg_NumOfReviewers_SameCategory',
                   'Population Within 1km', 'Population Within 2km', 'Popularity', 'District']

#training 
X_train = combined_df_train.drop(columns=columns_to_drop, errors='ignore')
y_train = combined_df_train[target_variable]

#testing
X_test = combined_df_test.drop(columns=columns_to_drop, errors='ignore')
y_test = combined_df_test[target_variable].values

# Optional: Print shape summary
print(f"Train set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")



Train set size: 1493
Test set size: 359


### Remove Highly Correlated features

In [8]:
# -----------------------------
# Step 2: Remove highly correlated features
# -----------------------------

def remove_highly_correlated_features(df, threshold=0.95):
    corr_matrix = df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    return df.drop(columns=to_drop), to_drop

X_train, removed_corr_features = remove_highly_correlated_features(X_train, threshold=0.95)
X_columns = X_train.columns
X_test = X_test[X_columns]

print(f"Removed highly correlated features: {removed_corr_features}")


Removed highly correlated features: ['Competition - Food & Dining/POI Density']


In [9]:
X_train.columns

Index(['generalCategory', 'Religious Institutions', 'Coffee Shops',
       'Food & Dining', 'Restaurants', 'Home & Construction Services',
       'Entertainment & Recreation', 'Retail & Shopping', 'Finance & Services',
       'Education', 'Health', 'Public & Government Services',
       'Hotels & Hospitality', 'Transportation & Travel', 'Beauty & Wellness',
       'POI Density', 'Avg Rating - Food & Dining', 'Population Within 3km',
       'Residential Average Price', 'Avg Num of Reviewers - All POIs'],
      dtype='object')

### Transformation and Scaling

In [10]:
from sklearn.preprocessing import PowerTransformer, MinMaxScaler

# Initialize PowerTransformer for X
pt = PowerTransformer(method='yeo-johnson', standardize=False)  # Set standardize=False since we'll use MinMaxScaler next
scaler = MinMaxScaler()

# Fit on train and apply to both train and test
X_train_transformed = scaler.fit_transform(pt.fit_transform(X_train))
X_test_transformed = scaler.transform(pt.transform(X_test))

# Reshape y to 2D arrays (required by sklearn scalers)
y_train_reshaped = y_train.values.reshape(-1, 1)
y_test_reshaped = y_test.reshape(-1, 1)

# Initialize separate transformers for y
pt_y = PowerTransformer(method='yeo-johnson', standardize=False)
scaler_y = MinMaxScaler()

# Fit on training target and transform both y_train and y_test
y_train_transformed = scaler_y.fit_transform(pt_y.fit_transform(y_train_reshaped)).ravel()
y_test_transformed = scaler_y.transform(pt_y.transform(y_test_reshaped)).ravel()


# AI Models 

## Random Forest

### Recursive Feature Elimination (RFE) 

In [105]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

# Assuming these are your inputs
# x_train_transformed, y_train_transformed, x_test_transformed, y_test_transformed
# And the original feature names
feature_names = X_train.columns.tolist()  # Replace this with the list of feature names before transformation

# Store selected features and R² for each value
results = []

# Define feature counts to try
feature_counts = [11, 9, 7, 5]

# Define grid
param_grid = {
    "n_estimators": [10, 50, 100, 200],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

# Loop through different values of n_features_to_select
for n in feature_counts:
    print(f"\n>>> Running RFE with top {n} features...")

    rf = RandomForestRegressor(random_state=42)
    rfe = RFE(estimator=rf, n_features_to_select=n)
    rfe.fit(X_train_transformed, y_train_transformed)

    # Get selected feature indices and names
    selected_indices = np.where(rfe.support_)[0]
    selected_features = [feature_names[i] for i in selected_indices]
    print(f"Selected Features: {selected_features}")

    # Subset the transformed data
    x_train_selected = X_train_transformed[:, selected_indices]
    x_test_selected = X_test_transformed[:, selected_indices]

    # Grid search
    grid = GridSearchCV(
        RandomForestRegressor(random_state=42),
        param_grid,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )
    
    grid.fit(x_train_selected, y_train_transformed)

    # Evaluate on test set
    y_pred = grid.predict(x_test_selected)
    r2 = r2_score(y_test_transformed, y_pred)
    print(f"R² on test set: {r2:.3f}")
    print(f"Best Params: {grid.best_params_}")

    # Save results
    results.append({
        'n_features': n,
        'selected_features': selected_features,
        'r2_score': r2,
        'best_params': grid.best_params_
    })

# Convert to DataFrame for summary
df_results = pd.DataFrame(results)
print("\n=== Summary ===")
print(df_results)



>>> Running RFE with top 11 features...


Selected Features: ['generalCategory', 'Food & Dining', 'Restaurants', 'Retail & Shopping', 'Finance & Services', 'Hotels & Hospitality', 'Transportation & Travel', 'Avg Rating - Food & Dining', 'Population Within 3km', 'Residential Average Price', 'Avg Num of Reviewers - All POIs']


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


R² on test set: 0.170
Best Params: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 50}

>>> Running RFE with top 9 features...
Selected Features: ['generalCategory', 'Food & Dining', 'Restaurants', 'Retail & Shopping', 'Finance & Services', 'Avg Rating - Food & Dining', 'Population Within 3km', 'Residential Average Price', 'Avg Num of Reviewers - All POIs']


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


R² on test set: 0.209
Best Params: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 200}

>>> Running RFE with top 7 features...
Selected Features: ['generalCategory', 'Food & Dining', 'Retail & Shopping', 'Avg Rating - Food & Dining', 'Population Within 3km', 'Residential Average Price', 'Avg Num of Reviewers - All POIs']


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


R² on test set: 0.191
Best Params: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100}

>>> Running RFE with top 5 features...
Selected Features: ['Food & Dining', 'Retail & Shopping', 'Avg Rating - Food & Dining', 'Population Within 3km', 'Avg Num of Reviewers - All POIs']


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


R² on test set: 0.068
Best Params: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 100}

=== Summary ===
   n_features                                  selected_features  r2_score  \
0          11  [generalCategory, Food & Dining, Restaurants, ...  0.170494   
1           9  [generalCategory, Food & Dining, Restaurants, ...  0.208872   
2           7  [generalCategory, Food & Dining, Retail & Shop...  0.190974   
3           5  [Food & Dining, Retail & Shopping, Avg Rating ...  0.068084   

                                         best_params  
0  {'max_depth': 10, 'max_features': 'sqrt', 'min...  
1  {'max_depth': 10, 'max_features': 'sqrt', 'min...  
2  {'max_depth': 10, 'max_features': 'sqrt', 'min...  
3  {'max_depth': 10, 'max_features': 'sqrt', 'min...  


With group splitting 

In [106]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import ndcg_score
import pandas as pd
import numpy as np

def compute_nsd(true_labels, pred_labels, k=10):
    true_top_k = set(np.argsort(true_labels)[::-1][:k])
    pred_top_k = set(np.argsort(pred_labels)[::-1][:k])
    symmetric_diff = true_top_k.symmetric_difference(pred_top_k)
    return len(symmetric_diff) / (2 * k)

# Set your feature names list
feature_names = X_train.columns.tolist()

results = []
feature_counts = [11, 9, 7, 5]

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt"]
}

for n in feature_counts:
    print(f"\n>>> Running RFE with top {n} features...")

    rf = RandomForestRegressor(random_state=42)
    rfe = RFE(estimator=rf, n_features_to_select=n)
    rfe.fit(X_train_transformed, y_train_transformed)

    selected_indices = np.where(rfe.support_)[0]
    selected_features = [feature_names[i] for i in selected_indices]
    print(f"Selected Features: {selected_features}")

    x_train_selected = X_train_transformed[:, selected_indices]
    x_test_selected = X_test_transformed[:, selected_indices]

    grid = GridSearchCV(
        RandomForestRegressor(random_state=42),
        param_grid,
        cv=3,
        scoring='neg_mean_squared_error',  # proxy, not used for final metric
        n_jobs=-1
    )
    
    grid.fit(x_train_selected, y_train_transformed)

    y_pred = grid.predict(x_test_selected)

    ndcg = ndcg_score([y_test_transformed], [y_pred], k=10)
    nsd = compute_nsd(y_test_transformed, y_pred, k=10)

    print(f"nDCG@10: {ndcg:.3f}")
    print(f"nSD@10:  {nsd:.3f}")
    print(f"Best Params: {grid.best_params_}")

    results.append({
        'n_features': n,
        'selected_features': selected_features,
        'nDCG@10': ndcg,
        'nSD@10': nsd,
        'best_params': grid.best_params_
    })

df_results = pd.DataFrame(results)
print("\n=== Summary ===")
print(df_results)


>>> Running RFE with top 11 features...


Selected Features: ['generalCategory', 'Food & Dining', 'Restaurants', 'Retail & Shopping', 'Finance & Services', 'Hotels & Hospitality', 'Transportation & Travel', 'Avg Rating - Food & Dining', 'Population Within 3km', 'Residential Average Price', 'Avg Num of Reviewers - All POIs']
nDCG@10: 0.745
nSD@10:  0.600
Best Params: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 50}

>>> Running RFE with top 9 features...
Selected Features: ['generalCategory', 'Food & Dining', 'Restaurants', 'Retail & Shopping', 'Finance & Services', 'Avg Rating - Food & Dining', 'Population Within 3km', 'Residential Average Price', 'Avg Num of Reviewers - All POIs']
nDCG@10: 0.724
nSD@10:  0.700
Best Params: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}

>>> Running RFE with top 7 features...
Selected Features: ['generalCategory', 'Food & Dining', 'Retail & Shopping', 'Avg Rating - Food & D

### Features Importance

### Average 10 runs

In [107]:
import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error, ndcg_score
)
from sklearn.preprocessing import PowerTransformer, MinMaxScaler

selected_columns = ['Food & Dining', 'Retail & Shopping', 'Avg Rating - Food & Dining', 'Population Within 3km', 'Avg Num of Reviewers - All POIs']

selected_indices = [X_train.columns.get_loc(col) for col in selected_columns]

X_train_selected = X_train_transformed[:, selected_indices]
X_test_selected= X_test_transformed[:, selected_indices]



# Container for results
results = []

# Repeat experiment 10 times
for run in range(10):
    #print(f"\n==================== Run {run+1}/10 ====================")

    # -----------------------------
    # Step 3: Grid Search - Random Forest
    # -----------------------------
    # Generate a seed randomly

    #rng = np.random.RandomState()  # system-random seed
    #seed_value = rng.randint(0, 2**16 - 1)

    rf_model = RandomForestRegressor(random_state=None, max_depth =10, max_features = 'sqrt', min_samples_leaf = 1, min_samples_split = 2, n_estimators = 100)

# param_grid = {
#     'max_depth': [10],
#     'max_features': ['sqrt'], 
#     'min_samples_leaf': [2], 
#     'min_samples_split': [2], 
#     'n_estimators': [100]
#     }

#grid_search = GridSearchCV(rf_model, param_grid, scoring='r2', verbose=1, n_jobs=-1)
    rf_model.fit(X_train_selected, y_train_transformed)
    y_pred = rf_model.predict(X_test_selected)
    
    # -----------------------------
    # Step 4: Predict & Inverse Transform
    # -----------------------------

    #y_pred = pt_y.inverse_transform(scaler_y.inverse_transform(y_pred.reshape(-1, 1))).ravel()

    # -----------------------------
    # Step 5: Regression Metrics
    # -----------------------------
    r2 = r2_score(y_test_transformed, y_pred)
    mse = mean_squared_error(y_test_transformed, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_transformed, y_pred)

    # -----------------------------
    # Step 6: Ranking Metrics
    # -----------------------------
    k = 10
    true_ranks = y_test_transformed.argsort()[::-1]
    pred_ranks = y_pred.argsort()[::-1]

    true_relevance = np.expand_dims(y_test_transformed, axis=0)
    predicted_scores = np.expand_dims(y_pred, axis=0)
    ndcg = ndcg_score(true_relevance, predicted_scores, k=k)

    top_k_true = set(true_ranks[:k])
    top_k_pred = set(pred_ranks[:k])
    # Remove NaN values before computing
    #mask = ~np.isnan(top_k_true) & ~np.isnan(top_k_pred)  # Keep only valid values
    #tau, _ = kendalltau(np.array(top_k_true)[mask], np.array(top_k_pred)[mask])
    nSD = len(top_k_true.symmetric_difference(top_k_pred)) / (2 * k)
    # -----------------------------
    # Step 7: Log Results
    # -----------------------------
    results.append({
        'run': run + 1,
        'r2': r2,
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        f'ndcg@{k}': ndcg,
        f'nSD@{k}': nSD,
        #f'Kendall’s Tau:': tau,
    })

# -----------------------------
# Step 8: Summary Report
# -----------------------------
results_df = pd.DataFrame(results)
print("\n==================== Summary of 10 Runs ====================")
print(results_df)

print("\n=== Average Metrics Across 10 Runs ===")
print(results_df.drop(columns=['run']).mean().round(3))
print("Last run:")
print('top_k_true',top_k_true)
print('top_k_pred',top_k_pred)



==================== Summary of 10 Runs ====================
   run        r2       mse      rmse       mae   ndcg@10  nSD@10
0    1  0.051444  0.025072  0.158342  0.123771  0.501889     0.9
1    2  0.038243  0.025421  0.159440  0.124502  0.504255     0.9
2    3  0.050821  0.025089  0.158394  0.123310  0.479224     0.9
3    4  0.039444  0.025389  0.159340  0.124075  0.461447     1.0
4    5  0.043202  0.025290  0.159028  0.123920  0.498290     0.9
5    6  0.053046  0.025030  0.158208  0.123180  0.431936     1.0
6    7  0.041497  0.025335  0.159170  0.124263  0.445813     0.9
7    8  0.039802  0.025380  0.159310  0.124906  0.492670     0.8
8    9  0.043992  0.025269  0.158962  0.123799  0.473084     1.0
9   10  0.062868  0.024770  0.157385  0.122402  0.471251     0.9

=== Average Metrics Across 10 Runs ===
r2         0.046
mse        0.025
rmse       0.159
mae        0.124
ndcg@10    0.476
nSD@10     0.920
dtype: float64
Last run:
top_k_true {226, 259, 137, 201, 267, 205, 179, 125, 345,

## XGBoost

### Average 10 runs

In [13]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, ndcg_score
from sklearn.preprocessing import PowerTransformer, MinMaxScaler

# Container for results
results = []

# Repeat experiment 10 times
for run in range(10):
    #print(f"\n==================== Run {run+1}/10 ====================")

    # -----------------------------
    # Step 3: Grid Search - XGBoost
    # -----------------------------
    xgb_model = XGBRegressor(objective='reg:squarederror', random_state=None)

    # GridSearch
    params_grid = {
            'n_estimators': [50],
            'max_depth': [3],
            'learning_rate': [0.1],
            'subsample': [0.8],
            'colsample_bytree': [0.8]
    }

    grid_search = GridSearchCV(
        estimator=xgb_model,
        param_grid=params_grid,
        scoring='r2',
        verbose=0,
        n_jobs=-1,
        cv=5
    )

    grid_search.fit(X_train_transformed, y_train_transformed)

    # -----------------------------
    # Step 4: Predict & Inverse Transform
    # -----------------------------
    y_pred = grid_search.best_estimator_.predict(X_test_transformed)
    #y_pred = pt_y.inverse_transform(scaler_y.inverse_transform(y_pred.reshape(-1, 1))).ravel()

    # -----------------------------
    # Step 5: Regression Metrics
    # -----------------------------
    r2 = r2_score(y_test_transformed, y_pred)
    mse = mean_squared_error(y_test_transformed, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_transformed, y_pred)

    # -----------------------------
    # Step 6: Ranking Metrics
    # -----------------------------
    k = 10
    true_ranks = y_test_transformed.argsort()[::-1]
    pred_ranks = y_pred.argsort()[::-1]

    true_relevance = np.expand_dims(y_test_transformed, axis=0)
    predicted_scores = np.expand_dims(y_pred, axis=0)
    ndcg = ndcg_score(true_relevance, predicted_scores, k=k)

    top_k_true = set(true_ranks[:k])
    top_k_pred = set(pred_ranks[:k])
    nSD = len(top_k_true.symmetric_difference(top_k_pred)) / (2 * k)

    # -----------------------------
    # Step 7: Log Results
    # -----------------------------
    results.append({
        'run': run + 1,
        'r2': r2,
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        f'ndcg@{k}': ndcg,
        f'nSD@{k}': nSD
    })

# -----------------------------
# Step 8: Summary Report
# -----------------------------
results_df = pd.DataFrame(results)
print("\n==================== Summary of 10 Runs ====================")
print(results_df)


print("\n=== Average Metrics Across 10 Runs ===")
print(results_df.drop(columns=['run']).mean().round(3))



==================== Summary of 10 Runs ====================
   run        r2       mse      rmse       mae   ndcg@10  nSD@10
0    1  0.201772  0.021099  0.145254  0.112871  0.770011     0.5
1    2  0.201772  0.021099  0.145254  0.112871  0.770011     0.5
2    3  0.201772  0.021099  0.145254  0.112871  0.770011     0.5
3    4  0.201772  0.021099  0.145254  0.112871  0.770011     0.5
4    5  0.201772  0.021099  0.145254  0.112871  0.770011     0.5
5    6  0.201772  0.021099  0.145254  0.112871  0.770011     0.5
6    7  0.201772  0.021099  0.145254  0.112871  0.770011     0.5
7    8  0.201772  0.021099  0.145254  0.112871  0.770011     0.5
8    9  0.201772  0.021099  0.145254  0.112871  0.770011     0.5
9   10  0.201772  0.021099  0.145254  0.112871  0.770011     0.5

=== Average Metrics Across 10 Runs ===
r2         0.202
mse        0.021
rmse       0.145
mae        0.113
ndcg@10    0.770
nSD@10     0.500
dtype: float64


### Recursive Feature Elimination (RFE)

In [13]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, ndcg_score
from sklearn.feature_selection import RFE
from sklearn.preprocessing import PowerTransformer, MinMaxScaler

results = []

feature_counts = [11, 9, 7, 5]
X_train_transformed = pd.DataFrame(X_train_transformed, columns=X_train.columns)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=X_test.columns)

for n in feature_counts:
    print(f"\n>>> Running RFE with top {n} features...")

    xgb_model = XGBRegressor(objective='reg:squarederror', random_state=None)
    
    rfe = RFE(estimator=xgb_model, n_features_to_select=n)
    rfe.fit(X_train_transformed, y_train_transformed)

    selected_features = X_train.columns[rfe.support_].tolist()
    print(f"Selected Features: {selected_features}")

    run_metrics = []

    for run in range(10):
        print(f"\n==================== Run {run+1}/10 ====================")

        # Initialize the XGBRegressor model
        xgb_model = XGBRegressor(objective='reg:squarederror', random_state=None)

        # GridSearch
        param_grid = {
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 5, 6, 10],
            'learning_rate': [0.01, 0.1, 0.2, 0.3],
            'subsample': [0.8, 1],
            'colsample_bytree': [0.8, 1]
        }

        #grid_search = GridSearchCV(
        #estimator=xgb_model,
        #param_grid=param_grid,
        #scoring='r2',
        #verbose=0,
        #n_jobs=-1,
        #cv=5
        # )
        grid_search = GridSearchCV(XGBRegressor(objective='reg:squarederror', random_state=None),
                                   param_grid,
                                   cv=5,
                                   scoring='r2',
                                   n_jobs=-1)
        grid_search.fit(X_train_transformed[selected_features], y_train_transformed)

        
        best_params = grid_search.best_params_
        print(f"Best Parameters for Run {run+1}: {best_params}")

        y_pred = grid_search.best_estimator_.predict(X_test_transformed[selected_features])

        # Regression Metrics
        r2 = r2_score(y_test_transformed, y_pred)
        mse = mean_squared_error(y_test_transformed, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test_transformed, y_pred)

        # Ranking Metrics
        k = 10
        true_ranks = y_test_transformed.argsort()[::-1]
        pred_ranks = y_pred.argsort()[::-1]

        true_relevance = np.expand_dims(y_test_transformed, axis=0)
        predicted_scores = np.expand_dims(y_pred, axis=0)
        ndcg = ndcg_score(true_relevance, predicted_scores, k=k)

        top_k_true = set(true_ranks[:k])
        top_k_pred = set(pred_ranks[:k])
        nSD = len(top_k_true.symmetric_difference(top_k_pred)) / (2 * k)

        # Metrics of the current run
        run_metrics.append({
            'run': run+1,
            'r2': r2,
            'mse': mse,
            'rmse': rmse,
            'mae': mae,
            f'ndcg@{k}': ndcg,
            f'nSD@{k}': nSD,
            'best_params': best_params  
        })

    # Average metrics across the 10 runs for the current feature set
    avg_metrics = {
        'n_features': n,
        'r2': round(np.mean([metric['r2'] for metric in run_metrics]), 3),
        'mse': round(np.mean([metric['mse'] for metric in run_metrics]), 3),
        'rmse': round(np.mean([metric['rmse'] for metric in run_metrics]), 3),
        'mae': round(np.mean([metric['mae'] for metric in run_metrics]), 3),
        f'ndcg@{k}': round(np.mean([metric[f'ndcg@{k}'] for metric in run_metrics]), 3),
        f'nSD@{k}': round(np.mean([metric[f'nSD@{k}'] for metric in run_metrics]), 3),
        'selected_features': selected_features,  # Using the selected features from RFE
        'best_params': best_params  
    }

    print("\n=== Individual Run Metrics ===")
    for run_metric in run_metrics:
        print(run_metric)
    
    print("\n=== Average Metrics ===")
    print(avg_metrics)

    results.append(avg_metrics)

df_results = pd.DataFrame(results)

# Summary
print("\n=== Summary ===")
print(df_results)



>>> Running RFE with top 11 features...
Selected Features: ['generalCategory', 'Education', 'Health', 'Public & Government Services', 'Hotels & Hospitality', 'Transportation & Travel', 'Beauty & Wellness', 'Avg Rating - Food & Dining', 'Population Within 3km', 'Residential Average Price', 'Avg Num of Reviewers - All POIs']

==================== Run 1/10 ====================


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best Parameters for Run 1: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}

==================== Run 2/10 ====================
Best Parameters for Run 2: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}

==================== Run 3/10 ====================


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best Parameters for Run 3: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}

==================== Run 4/10 ====================


...

C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best Parameters for Run 8: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}

==================== Run 9/10 ====================


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best Parameters for Run 9: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}

==================== Run 10/10 ====================
Best Parameters for Run 10: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}

=== Individual Run Metrics ===
{'run': 1, 'r2': 0.15496636121729668, 'mse': 0.02233576205054002, 'rmse': 0.14945153746462436, 'mae': 0.11701491949770797, 'ndcg@10': 0.7910126655808397, 'nSD@10': 0.5, 'best_params': {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}}
{'run': 2, 'r2': 0.15496636121729668, 'mse': 0.02233576205054002, 'rmse': 0.14945153746462436, 'mae': 0.11701491949770797, 'ndcg@10': 0.7910126655808397, 'nSD@10': 0.5, 'best_params': {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}}
{'run': 3, 'r2': 0.15496636121729668, 'mse': 0.02233576205054002, 'rmse': 0.149451

C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


### Exhaustive Feature Selection (EFS)

In [14]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, ndcg_score
from mlxtend.feature_selection import ExhaustiveFeatureSelector as EFS
from sklearn.preprocessing import PowerTransformer, MinMaxScaler

results = []

feature_counts = [9, 11]
X_train_transformed = pd.DataFrame(X_train_transformed, columns=X_train.columns)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=X_test.columns)

for n in feature_counts:
    print(f"\n>>> Running Exhaustive Feature Selection with top {n} features...")

    xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)

    efs = EFS(
        estimator=xgb_model,
        min_features=n,
        max_features=n,
        scoring='r2',
        print_progress=True,
        cv=5,
        n_jobs=-1
    )

    efs.fit(X_train_transformed, y_train_transformed)

    selected_features = list(efs.best_feature_names_)
    print(f"Selected Features: {selected_features}")

    # GridSearch for best hyperparameters
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 6, 10],
        'learning_rate': [0.01, 0.1, 0.2, 0.3],
        'subsample': [0.8, 1],
        'colsample_bytree': [0.8, 1]
    }

    grid_search = GridSearchCV(
        XGBRegressor(objective='reg:squarederror', random_state=42),
        param_grid,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )

    grid_search.fit(X_train_transformed[selected_features], y_train_transformed)

    best_params = grid_search.best_params_
    print(f"Best Parameters: {best_params}")

    y_pred = grid_search.best_estimator_.predict(X_test_transformed[selected_features])

    # Regression Metrics
    r2 = r2_score(y_test_transformed, y_pred)
    mse = mean_squared_error(y_test_transformed, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_transformed, y_pred)

    # Ranking Metrics
    k = 10
    true_relevance = np.expand_dims(y_test_transformed, axis=0)
    predicted_scores = np.expand_dims(y_pred, axis=0)
    ndcg = ndcg_score(true_relevance, predicted_scores, k=k)

    true_ranks = y_test_transformed.argsort()[::-1]
    pred_ranks = y_pred.argsort()[::-1]
    top_k_true = set(true_ranks[:k])
    top_k_pred = set(pred_ranks[:k])
    nSD = len(top_k_true.symmetric_difference(top_k_pred)) / (2 * k)

    avg_metrics = {
        'n_features': n,
        'r2': round(r2, 3),
        'mse': round(mse, 3),
        'rmse': round(rmse, 3),
        'mae': round(mae, 3),
        f'ndcg@{k}': round(ndcg, 3),
        f'nSD@{k}': round(nSD, 3),
        'selected_features': selected_features,
        'best_params': best_params
    }

    print("\n=== Metrics ===")
    print(avg_metrics)

    results.append(avg_metrics)

df_results = pd.DataFrame(results)

# Summary
print("\n=== Summary ===")
print(df_results)


>>> Running Exhaustive Feature Selection with top 9 features...


Features: 167960/167960

Selected Features: ['generalCategory', 'Religious Institutions', 'Retail & Shopping', 'Finance & Services', 'Hotels & Hospitality', 'Beauty & Wellness', 'POI Density', 'Population Within 3km', 'Avg Num of Reviewers - All POIs']
Best Parameters: {'colsample_bytree': 1, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}

=== Metrics ===
{'n_features': 9, 'r2': 0.184, 'mse': 0.022, 'rmse': 0.147, 'mae': 0.114, 'ndcg@10': 0.729, 'nSD@10': 0.8, 'selected_features': ['generalCategory', 'Religious Institutions', 'Retail & Shopping', 'Finance & Services', 'Hotels & Hospitality', 'Beauty & Wellness', 'POI Density', 'Population Within 3km', 'Avg Num of Reviewers - All POIs'], 'best_params': {'colsample_bytree': 1, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}}

>>> Running Exhaustive Feature Selection with top 11 features...


## SVR 

### Recursive Feature Elimination (RFE) 

In [110]:
from sklearn.feature_selection import RFE
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

# Assuming these are your inputs
# x_train_transformed, y_train_transformed, x_test_transformed, y_test_transformed
# And the original feature names
feature_names = X_train.columns.tolist()  # Replace this with the list of feature names before transformation

# Store selected features and R² for each value
results = []

# Define feature counts to try
feature_counts = [11, 9, 7, 5]

# Define grid for SVR
param_grid = {
    "kernel": ['linear', 'rbf', 'sigmoid'],
    "C": [0.1, 1, 10, 100],
    "gamma": ['scale', 0.01, 0.001, 0.1, 0.2, 0.3],
    "epsilon": [0.01, 0.1, 0.2]
}

# Loop through different values of n_features_to_select
for n in feature_counts:
    print(f"\n>>> Running RFE with top {n} features...")

    svr = SVR(kernel='linear')
    rfe = RFE(estimator=svr, n_features_to_select=n)
    rfe.fit(X_train_transformed, y_train_transformed)

    # Get selected feature indices and names
    selected_indices = np.where(rfe.support_)[0]
    selected_features = [feature_names[i] for i in selected_indices]
    print(f"Selected Features: {selected_features}")

    # Subset the transformed data
    x_train_selected = X_train_transformed[:, selected_indices]
    x_test_selected = X_test_transformed[:, selected_indices]

    # Grid search with SVR
    grid = GridSearchCV(
        SVR(),
        param_grid,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )
    
    grid.fit(x_train_selected, y_train_transformed)

    # Evaluate on test set
    y_pred = grid.predict(x_test_selected)
    r2 = r2_score(y_test_transformed, y_pred)
    print(f"R² on test set: {r2:.3f}")
    print(f"Best Params: {grid.best_params_}")

    # Save results
    results.append({
        'n_features': n,
        'selected_features': selected_features,
        'r2_score': r2,
        'best_params': grid.best_params_
    })

# Convert to DataFrame for summary
df_results = pd.DataFrame(results)
print("\n=== Summary ===")
print(df_results)



>>> Running RFE with top 11 features...
Selected Features: ['generalCategory', 'Coffee Shops', 'Retail & Shopping', 'Finance & Services', 'Education', 'Transportation & Travel', 'Beauty & Wellness', 'Avg Rating - Food & Dining', 'Population Within 3km', 'Residential Average Price', 'Avg Num of Reviewers - All POIs']


InvalidIndexError: (slice(None, None, None), array([ 0,  2,  7,  8,  9, 13, 14, 16, 17, 18, 19], dtype=int64))

### Average 10 runs

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    ndcg_score
)



selected_columns = ['generalCategory', 'Retail & Shopping', 'Avg Rating - Food & Dining', 'Population Within 3km', 'Avg Num of Reviewers - All POIs']

selected_indices = [X_train.columns.get_loc(col) for col in selected_columns]

X_train_selected = X_train_transformed[:, selected_indices]
X_test_selected= X_test_transformed[:, selected_indices]


# Placeholder for logging results
results = []

for run in range(10):
   # print(f"\n==================== Run {run+1}/10 ====================")

    # -----------------------------
    # Step 3: Define SVR + Grid Search
    # -----------------------------
    model = Pipeline([('svr', SVR())])

    param_grid = {
        'svr__C': [10],
        'svr__gamma': [0.2],
        'svr__epsilon': [0.2],
        'svr__kernel': ['rbf']
    }

    #param_grid = {
       # 'svr__C': [0.01, 0.1, 1, 10],
        #'svr__gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 2],
        #'svr__epsilon': [0.001, 0.01, 0.05, 0.1],
       # 'svr__kernel': ['rbf', 'sigmoid']
   # }
    grid_search = GridSearchCV(model, param_grid, scoring="r2", cv=5, n_jobs=-1, verbose=0)
    grid_search.fit(X_train_selected, y_train_transformed)

    # -----------------------------
    # Step 4: Predict & Inverse Transform
    # -----------------------------
    y_pred = grid_search.best_estimator_.predict(X_test_selected)
    #y_pred = pt_y.inverse_transform(scaler_y.inverse_transform(y_pred.reshape(-1, 1))).ravel()

    # -----------------------------
    # Step 5: Regression Metrics
    # -----------------------------
    r2 = r2_score(y_test_transformed, y_pred)
    mse = mean_squared_error(y_test_transformed, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_transformed, y_pred)

    # -----------------------------
    # Step 6: Ranking Metrics
    # -----------------------------
    k = 10
    true_ranks = y_test_transformed.argsort()[::-1]
    pred_ranks = y_pred.argsort()[::-1]

    true_relevance = np.expand_dims(y_test_transformed, axis=0)
    predicted_scores = np.expand_dims(y_pred, axis=0)
    ndcg = ndcg_score(true_relevance, predicted_scores, k=k)

    top_k_true = set(true_ranks[:k])
    top_k_pred = set(pred_ranks[:k])
    nSD = len(top_k_true.symmetric_difference(top_k_pred)) / (2 * k)

    # -----------------------------
    # Step 7: Log Results
    # -----------------------------
    results.append({
        'run': run + 1,
        #'best_params': grid_search.best_params_,
        'r2': r2,
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        f'ndcg@{k}': ndcg,
        f'nSD@{k}': nSD
    })

# -----------------------------
# Step 8: Summary Report
# -----------------------------
results_df = pd.DataFrame(results)
print("\n==================== Summary of 10 Runs ====================")
print(results_df)

print("\n=== Average Metrics Across 10 Runs ===")
print(results_df.drop(columns=['run']).mean().round(3))



==================== Summary of 10 Runs ====================
   run        r2       mse      rmse       mae   ndcg@10  nSD@10
0    1  0.214419  0.020764  0.144098  0.113319  0.725486     0.7
1    2  0.214419  0.020764  0.144098  0.113319  0.725486     0.7
2    3  0.214419  0.020764  0.144098  0.113319  0.725486     0.7
3    4  0.214419  0.020764  0.144098  0.113319  0.725486     0.7
4    5  0.214419  0.020764  0.144098  0.113319  0.725486     0.7
5    6  0.214419  0.020764  0.144098  0.113319  0.725486     0.7
6    7  0.214419  0.020764  0.144098  0.113319  0.725486     0.7
7    8  0.214419  0.020764  0.144098  0.113319  0.725486     0.7
8    9  0.214419  0.020764  0.144098  0.113319  0.725486     0.7
9   10  0.214419  0.020764  0.144098  0.113319  0.725486     0.7

=== Average Metrics Across 10 Runs ===
r2         0.214
mse        0.021
rmse       0.144
mae        0.113
ndcg@10    0.725
nSD@10     0.700
dtype: float64


## LambdaMART

### All features

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer, MinMaxScaler
from sklearn.metrics import ndcg_score
import lightgbm as lgb

def compute_nsd(true_labels, pred_labels, k=10):
    """ Compute Normalized Symmetric Difference at k (nSD@k) """
    true_top_k = set(np.argsort(true_labels)[::-1][:k])
    pred_top_k = set(np.argsort(pred_labels)[::-1][:k])
    symmetric_diff = true_top_k.symmetric_difference(pred_top_k)
    return len(symmetric_diff) / (2 * k)

# ---- 1. Load data ----
df = pd.read_csv("../../Cleaned Data/Cleaned_Data_LastVersion.csv")

# Ensure that the 'Number of Reviewers' column is numeric and handle errors as NaN
df['Number of Reviewers'] = pd.to_numeric(df['Number of Reviewers'], errors='coerce')
# Drop rows where 'Number of Reviewers' is NaN (if you want to ignore them in analysis)
df = df.dropna(subset=['Number of Reviewers'])

drop_columns = ['Business ID', 'Name', 'Category', 'Rating', 'Google Place ID', 'Business Status',
                'Cluster', 'Distance (m)', 'Avg Rating - Business Type', 'Competition - Business Type/Area', 
                'Competition - Food & Dining/Area', 'Competition - Business Type/POI Density', 
                'Competition - Food & Dining/related POIs', 'Competition - Business Type/related POIs',
                'Google Rating', 'Business Name', 'Popularity', 'Latitude', 'Longitude', 
                'Total Num of Reviewers - Food & Dining', 'Total Num of Reviewers - All POIs', 
                'Avg_NumOfReviewers - Food & Dining','Avg_NumOfReviewers_SameCategory', 
                'Total Num of Reviewers - Food & Dining','Population Within 1km', 
                'Population Within 2km', 'Popularity', 'Competition - Food & Dining/POI Density', 'Residential Average Price']

df.drop(columns=drop_columns, inplace=True, errors='ignore')
                # 'Health',  
#                 # 'Home & Construction Services', 'generalCategory', 'Religious Institutions', 'Hotels & Hospitality', 
#                 # 'Restaurants', 'Food & Dining' , 'Population Within 3km' ]

# ---- 2. Feature columns ----
feature_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
feature_cols = [col for col in feature_cols if col != 'Number of Reviewers']
df = df.dropna(subset=['Number of Reviewers'] + feature_cols).reset_index(drop=True)

# ---- 3. Split districts into train/test ----
districts = df['District'].unique()
train_districts, test_districts = train_test_split(districts, test_size=0.2, random_state=42)
train_df = df[df['District'].isin(train_districts)].reset_index(drop=True)
test_df = df[df['District'].isin(test_districts)].reset_index(drop=True)

# ---- 4. Relevance score binning ----
y_train = train_df['Number of Reviewers']
y_test = test_df['Number of Reviewers']
y_train_relevance = pd.qcut(y_train, q=5, labels=False, duplicates='drop')
y_test_relevance = pd.qcut(y_test, q=5, labels=False, duplicates='drop')

# ---- 5. Feature transformation ----
pt = PowerTransformer(method='yeo-johnson', standardize=False)
scaler = MinMaxScaler()
X_train_base = pt.fit_transform(train_df[feature_cols])
X_train_base = scaler.fit_transform(X_train_base)
X_test_base = pt.transform(test_df[feature_cols])
X_test_base = scaler.transform(X_test_base)

train_groups = train_df.groupby('District').size().values
test_groups = test_df.groupby('District').size().values

# ---- 6. Grid Search ----
param_grid = {
    'num_leaves': [15, 31, 63],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth':        [-1, 5, 10, 15],
    'min_data_in_leaf': [5, 10, 20],
    'min_gain_to_split': [0.1, 0.5, 1.0], 
}

best_score = -1
best_model = None
best_params = None

for leaves in param_grid['num_leaves']:
    for lr in param_grid['learning_rate']:
        params = {
            'objective': 'lambdarank',
            'metric': 'ndcg',
            'boosting_type': 'gbdt',
            'num_leaves': leaves,
            'learning_rate': lr,
            'verbosity': -1
        }

        train_data = lgb.Dataset(X_train_base, label=y_train_relevance, group=train_groups)
        test_data = lgb.Dataset(X_test_base, label=y_test_relevance, group=test_groups)

        model = lgb.train(
            params,
            train_set=train_data,
            valid_sets=[test_data],
            num_boost_round=100,
            callbacks=[lgb.early_stopping(stopping_rounds=10)],
        )

        y_pred = model.predict(X_test_base)
        ndcg = ndcg_score([y_test_relevance], [y_pred], k=10)
        nsd_at_10 = compute_nsd(y_test_relevance, y_pred, k=10)
        print(f"[Grid] num_leaves={leaves}, learning_rate={lr} -> nDCG@10: {ndcg:.3f}, nSD@10: {nsd_at_10:.3f}")

        if ndcg > best_score:
            best_score = ndcg
            best_params = params
            best_model = model

print("\n Best Grid Params:", best_params)
print(" Best nDCG@10 from Grid Search:", best_score)

# ---- 7. Feature Importance ----
print("\n--- Feature Importance (Gain) ---")
importance = best_model.feature_importance(importance_type='gain')
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importance
}).sort_values(by='Importance', ascending=False)

print(importance_df.to_string(index=False))
print(feature_cols)

Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[11]	valid_0's ndcg@1: 0.69707	valid_0's ndcg@2: 0.679992	valid_0's ndcg@3: 0.686979	valid_0's ndcg@4: 0.693567	valid_0's ndcg@5: 0.691143
[Grid] num_leaves=15, learning_rate=0.01 -> nDCG@10: 0.550, nSD@10: 1.000
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[4]	valid_0's ndcg@1: 0.668132	valid_0's ndcg@2: 0.690342	valid_0's ndcg@3: 0.689552	valid_0's ndcg@4: 0.687131	valid_0's ndcg@5: 0.699828
[Grid] num_leaves=15, learning_rate=0.05 -> nDCG@10: 0.550, nSD@10: 1.000
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[9]	valid_0's ndcg@1: 0.72967	valid_0's ndcg@2: 0.712203	valid_0's ndcg@3: 0.731204	valid_0's ndcg@4: 0.739069	valid_0's ndcg@5: 0.756587
[Grid] num_leaves=15, learning_rate=0.1 -> nDCG@10: 0.662, nSD@10: 0.800
Training until validation scores don't improve for 10 rounds
Early stopping

In [12]:
# Compute and print variance of all numeric columns
df = pd.read_csv("../../Cleaned Data/Cleaned_Data_LastVersion.csv")
drop_columns = ['Business ID', 'Name', 'Category', 'Rating', 'Google Place ID', 'Business Status',
                'Cluster', 'Distance (m)', 'Avg Rating - Business Type', 'Competition - Business Type/Area', 
                'Competition - Food & Dining/Area', 'Competition - Business Type/POI Density', 
                'Competition - Food & Dining/related POIs', 'Competition - Business Type/related POIs',
                'Google Rating', 'Business Name', 'Popularity', 'Latitude', 'Longitude', 
                'Total Num of Reviewers - Food & Dining', 'Total Num of Reviewers - All POIs', 
                'Avg_NumOfReviewers - Food & Dining','Avg_NumOfReviewers_SameCategory', 
                'Total Num of Reviewers - Food & Dining','Population Within 1km', 
                'Population Within 2km', 'Popularity', 'Competition - Food & Dining/POI Density']
df.drop(columns=drop_columns, inplace=True, errors='ignore')

# ---- 3. Split districts into train/test ----
districts = df['District'].unique()
train_districts, test_districts = train_test_split(districts, test_size=0.2, random_state=42)
train_df = df[df['District'].isin(train_districts)].reset_index(drop=True)
test_df = df[df['District'].isin(test_districts)].reset_index(drop=True)
variances = train_df.var(numeric_only=True)
print("Feature Variance:\n")
print(variances.sort_values(ascending=False))

Feature Variance:

Population Within 3km              9.132115e+09
Residential Average Price          1.756570e+08
Number of Reviewers                2.531690e+06
Avg Num of Reviewers - All POIs    6.646707e+04
Retail & Shopping                  5.035172e+01
Food & Dining                      3.644314e+01
Restaurants                        2.140946e+01
Hotels & Hospitality               1.974008e+01
Coffee Shops                       7.758134e+00
Transportation & Travel            6.475597e+00
Beauty & Wellness                  5.148645e+00
POI Density                        4.828332e+00
Finance & Services                 4.235099e+00
Health                             2.982218e+00
Entertainment & Recreation         2.358578e+00
Education                          2.349169e+00
Religious Institutions             1.214247e+00
Public & Government Services       8.739019e-01
generalCategory                    2.316262e-01
Home & Construction Services       2.157616e-01
Avg Rating - Food & D

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv("../../Cleaned Data/Cleaned_Data_LastVersion.csv")

# Drop unnecessary columns
drop_columns = ['Business ID', 'Name', 'Category', 'Rating', 'Google Place ID', 'Business Status',
                'Cluster', 'Distance (m)', 'Avg Rating - Business Type', 'Competition - Business Type/Area', 
                'Competition - Food & Dining/Area', 'Competition - Business Type/POI Density', 
                'Competition - Food & Dining/related POIs', 'Competition - Business Type/related POIs',
                'Google Rating', 'Business Name', 'Popularity', 'Latitude', 'Longitude', 
                'Total Num of Reviewers - Food & Dining', 'Total Num of Reviewers - All POIs', 
                'Avg_NumOfReviewers - Food & Dining','Avg_NumOfReviewers_SameCategory', 
                'Total Num of Reviewers - Food & Dining','Population Within 1km', 
                'Population Within 2km', 'Popularity', 'Competition - Food & Dining/POI Density']
df.drop(columns=drop_columns, inplace=True, errors='ignore')

# Split districts into train/test
districts = df['District'].unique()
train_districts, test_districts = train_test_split(districts, test_size=0.2, random_state=42)
train_df = df[df['District'].isin(train_districts)].reset_index(drop=True)

# Select only numeric columns
numeric_cols = train_df.select_dtypes(include=['float64', 'int64']).columns

# Group by district and compute variance
print("\n📊 Variance per District (Training Set):")
district_variances = train_df.groupby('District')[numeric_cols].var()

# Print nicely
for district, row in district_variances.iterrows():
    print(f"\nDistrict: {district}")
    print(row.sort_values(ascending=False))



📊 Variance per District (Training Set):

District: Ad Dirah
Number of Reviewers                421756.250000
Population Within 3km                6642.250000
Avg Num of Reviewers - All POIs      1218.133682
Retail & Shopping                      41.583333
Beauty & Wellness                      11.583333
Transportation & Travel                 4.250000
Food & Dining                           1.666667
Finance & Services                      1.583333
Religious Institutions                  0.916667
Hotels & Hospitality                    0.916667
Health                                  0.916667
Restaurants                             0.916667
Coffee Shops                            0.666667
POI Density                             0.250000
Public & Government Services            0.250000
generalCategory                         0.250000
Education                               0.250000
Avg Rating - Food & Dining              0.014858
Entertainment & Recreation              0.000000
Home & C

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv("../../Cleaned Data/Cleaned_Data_LastVersion.csv")

# Keep only the needed columns
columns_to_keep = ['District', 'Population Within 3km']
df = df[columns_to_keep].dropna()

# Split districts into train/test
districts = df['District'].unique()
train_districts, test_districts = train_test_split(districts, test_size=0.2, random_state=42)
train_df = df[df['District'].isin(train_districts)]

# Group by district and compute variance for "Population Within 3km"
population_var_per_district = train_df.groupby('District')['Population Within 3km'].var()

# Compute the average of these per-district variances
average_population_variance = population_var_per_district.mean()

print("📊 Variance per District for 'Population Within 3km':\n")
print(population_var_per_district.sort_values(ascending=False))

print(f"\n📈 Average Variance Across Districts: {average_population_variance:.2f}")


📊 Variance per District for 'Population Within 3km':

District
Al Khalij              3.196841e+10
Ghirnatah              2.166018e+10
Al Kurnaish            6.012110e+09
Al Hamra               4.942326e+09
Al Sahil               4.693043e+09
                           ...     
An Nasim                        NaN
Ar Rabi                         NaN
Ibn Sina                        NaN
Sinaiyah Al Thuqbah             NaN
Sports City                     NaN
Name: Population Within 3km, Length: 101, dtype: float64

📈 Average Variance Across Districts: 1338660466.18


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv("../../Cleaned Data/Cleaned_Data_LastVersion.csv")

# Drop unnecessary columns
drop_columns = ['Business ID', 'Name', 'Category', 'Rating', 'Google Place ID', 'Business Status',
                'Cluster', 'Distance (m)', 'Avg Rating - Business Type', 'Competition - Business Type/Area', 
                'Competition - Food & Dining/Area', 'Competition - Business Type/POI Density', 
                'Competition - Food & Dining/related POIs', 'Competition - Business Type/related POIs',
                'Google Rating', 'Business Name', 'Popularity', 'Latitude', 'Longitude', 
                'Total Num of Reviewers - Food & Dining', 'Total Num of Reviewers - All POIs', 
                'Avg_NumOfReviewers - Food & Dining','Avg_NumOfReviewers_SameCategory', 
                'Total Num of Reviewers - Food & Dining','Population Within 1km', 
                'Population Within 2km', 'Popularity', 'Competition - Food & Dining/POI Density']
df.drop(columns=drop_columns, inplace=True, errors='ignore')

# Split into training districts
districts = df['District'].unique()
train_districts, _ = train_test_split(districts, test_size=0.2, random_state=42)
train_df = df[df['District'].isin(train_districts)].copy()

# Get numeric features
numeric_cols = train_df.select_dtypes(include=['float64', 'int64']).columns

# Compute variance per district for each numeric feature
district_variances = train_df.groupby('District')[numeric_cols].var()

# Compute the average variance for each feature
average_variances = district_variances.mean().sort_values(ascending=False)

print("📊 Average Per-District Variance for Each Feature (Training Set):\n")
print(average_variances)


📊 Average Per-District Variance for Each Feature (Training Set):

Population Within 3km              1.338660e+09
Number of Reviewers                1.628124e+06
Residential Average Price          1.318898e+05
Avg Num of Reviewers - All POIs    2.019651e+04
Retail & Shopping                  1.950123e+01
Food & Dining                      1.407955e+01
Restaurants                        9.175782e+00
POI Density                        4.431780e+00
Hotels & Hospitality               3.784758e+00
Transportation & Travel            3.724783e+00
Coffee Shops                       2.949622e+00
Beauty & Wellness                  2.806632e+00
Finance & Services                 1.981101e+00
Health                             1.503386e+00
Education                          1.289373e+00
Entertainment & Recreation         1.113916e+00
Religious Institutions             8.477731e-01
Public & Government Services       4.585671e-01
generalCategory                    1.937205e-01
Home & Construction Se

### After dropping features 

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer, MinMaxScaler
from sklearn.metrics import ndcg_score
import lightgbm as lgb

def compute_nsd(true_labels, pred_labels, k=10):
    """ Compute Normalized Symmetric Difference at k (nSD@k) """
    true_top_k = set(np.argsort(true_labels)[::-1][:k])
    pred_top_k = set(np.argsort(pred_labels)[::-1][:k])
    symmetric_diff = true_top_k.symmetric_difference(pred_top_k)
    return len(symmetric_diff) / (2 * k)

# ---- 1. Load data ----
df = pd.read_csv("../../Cleaned Data/Cleaned_Data_LastVersion.csv")

drop_columns = ['Business ID', 'Name', 'Category', 'Rating', 'Google Place ID', 'Business Status',
                'Cluster', 'Distance (m)', 'Avg Rating - Business Type', 'Competition - Business Type/Area', 
                'Competition - Food & Dining/Area', 'Competition - Business Type/POI Density', 
                'Competition - Food & Dining/related POIs', 'Competition - Business Type/related POIs',
                'Google Rating', 'Business Name', 'Popularity', 'Latitude', 'Longitude', 
                'Total Num of Reviewers - Food & Dining', 'Total Num of Reviewers - All POIs', 
                'Avg_NumOfReviewers - Food & Dining','Avg_NumOfReviewers_SameCategory', 
                'Total Num of Reviewers - Food & Dining','Population Within 1km', 
                'Population Within 2km', 'Popularity', 'Competition - Food & Dining/POI Density', 'Residential Average Price', 'Education', 'Food & Dining ','Entertainment & Recreation',' Home & Construction Services',
                'Finance & Services', 'Public & Government Services', 'POI Density', 'Coffee Shops', 'Residential Average Price', 'Health'] 
                # 'Home & Construction Services', 'generalCategory', 'Religious Institutions', 'Hotels & Hospitality', 
                # 'Restaurants', 'Food & Dining' , 'Population Within 3km' ]
                

df.drop(columns=drop_columns, inplace=True, errors='ignore')

# ---- 2. Feature columns ----
feature_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
feature_cols = [col for col in feature_cols if col != 'Number of Reviewers']
df = df.dropna(subset=['Number of Reviewers'] + feature_cols).reset_index(drop=True)

# ---- 3. Split districts into train/test ----
districts = df['District'].unique()
train_districts, test_districts = train_test_split(districts, test_size=0.2, random_state=42)
train_df = df[df['District'].isin(train_districts)].reset_index(drop=True)
test_df = df[df['District'].isin(test_districts)].reset_index(drop=True)

# ---- 4. Relevance score binning ----
y_train = train_df['Number of Reviewers']
y_test = test_df['Number of Reviewers']
y_train_relevance = pd.qcut(y_train, q=5, labels=False, duplicates='drop')
y_test_relevance = pd.qcut(y_test, q=5, labels=False, duplicates='drop')

# ---- 5. Feature transformation ----
pt = PowerTransformer(method='yeo-johnson', standardize=False)
scaler = MinMaxScaler()
X_train_base = pt.fit_transform(train_df[feature_cols])
X_train_base = scaler.fit_transform(X_train_base)
X_test_base = pt.transform(test_df[feature_cols])
X_test_base = scaler.transform(X_test_base)

train_groups = train_df.groupby('District').size().values
test_groups = test_df.groupby('District').size().values

# ---- 6. Grid Search ----
param_grid = {
    'num_leaves': [15, 31, 63],
    'learning_rate': [0.01, 0.05, 0.1],
}

best_score = -1
best_model = None
best_params = None

for leaves in param_grid['num_leaves']:
    for lr in param_grid['learning_rate']:
        params = {
            'objective': 'lambdarank',
            'metric': 'ndcg',
            'boosting_type': 'gbdt',
            'num_leaves': leaves,
            'learning_rate': lr,
            'verbosity': -1
        }

        train_data = lgb.Dataset(X_train_base, label=y_train_relevance, group=train_groups)
        test_data = lgb.Dataset(X_test_base, label=y_test_relevance, group=test_groups)

        model = lgb.train(
            params,
            train_set=train_data,
            valid_sets=[test_data],
            num_boost_round=100,
            callbacks=[lgb.early_stopping(stopping_rounds=10)],
        )

        y_pred = model.predict(X_test_base)
        ndcg = ndcg_score([y_test_relevance], [y_pred], k=10)
        nsd_at_10 = compute_nsd(y_test_relevance, y_pred, k=10)
        print(f"[Grid] num_leaves={leaves}, learning_rate={lr} -> nDCG@10: {ndcg:.3f}, nSD@10: {nsd_at_10:.3f}")

        if ndcg > best_score:
            best_score = ndcg
            best_params = params
            best_model = model

print("\n Best Grid Params:", best_params)
print(" Best nDCG@10 from Grid Search:", best_score)

# ---- 7. Feature Importance ----
print("\n--- Feature Importance (Gain) ---")
importance = best_model.feature_importance(importance_type='gain')
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importance
}).sort_values(by='Importance', ascending=False)

print(importance_df.to_string(index=False))
print(feature_cols)

Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[10]	valid_0's ndcg@1: 0.647619	valid_0's ndcg@2: 0.646023	valid_0's ndcg@3: 0.649978	valid_0's ndcg@4: 0.664526	valid_0's ndcg@5: 0.670315
[Grid] num_leaves=15, learning_rate=0.01 -> nDCG@10: 0.458, nSD@10: 1.000
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[21]	valid_0's ndcg@1: 0.680952	valid_0's ndcg@2: 0.650735	valid_0's ndcg@3: 0.661624	valid_0's ndcg@4: 0.695849	valid_0's ndcg@5: 0.711017
[Grid] num_leaves=15, learning_rate=0.05 -> nDCG@10: 0.535, nSD@10: 1.000
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[11]	valid_0's ndcg@1: 0.721978	valid_0's ndcg@2: 0.712591	valid_0's ndcg@3: 0.712728	valid_0's ndcg@4: 0.711687	valid_0's ndcg@5: 0.723723
[Grid] num_leaves=15, learning_rate=0.1 -> nDCG@10: 0.654, nSD@10: 1.000
Training until validation scores don't improve for 10 rounds
Early stop